# 🚗 Treinamento do YOLO com o Dataset Oficial UFPR-ALPR (Placas Brasileiras)
### Dataset: UFPR-ALPR (Universidade Federal do Paraná - 4.500 imagens anotadas de veículos brasileiros)

Este notebook automatiza:
1. Download Turbo (Multithread) do dataset oficial de 9.07 GB com `aria2c`
2. Descompactação e conversão automática de 100% das anotações de placas para o padrão YOLO (Divisão Treino/Validação 80/20)
3. Treinamento do modelo YOLO com GPU T4 (Transfer Learning)
4. Avaliação com métricas de validação em placas brasileiras (Mercosul e Antigas)
5. Download dos pesos treinados (`best.pt`) para o seu projeto local

## 1. Verificação da GPU
Certifique-se de que a GPU está ativada no Colab em: **Ambiente de execução > Alterar tipo de ambiente de execução > GPU T4**.

In [ ]:
!nvidia-smi

## 2. Instalação das Dependências

In [ ]:
!pip install -q ultralytics pillow
print("✅ Ultralytics instalado com sucesso!")

## 3. Download Turbo (Multithread) e Descompactação do Dataset UFPR-ALPR (9.07 GB)
Utilizamos o `aria2c` com 16 conexões simultâneas para baixar o arquivo de 9.07 GB na velocidade máxima da rede do Google Colab.

In [ ]:
import os

ZIP_DESTINO = "/content/UFPR-ALPR.zip"
PASTA_EXTRACAO = "/content/ufpr_raw"

# 1. Instala o acelerador de download aria2
!apt-get install -y aria2 > /dev/null 2>&1

# 2. Baixa com 16 conexões simultâneas de alta velocidade
if not os.path.exists(ZIP_DESTINO):
    print("🚀 Baixando dataset oficial UFPR-ALPR (9.07 GB) em alta velocidade...")
    !aria2c -x 16 -s 16 -j 16 -k 1M "https://www.inf.ufpr.br/vri/databases/yj4Iu2-UFPR-ALPR.zip" -d /content -o UFPR-ALPR.zip
    print("✅ Download concluído com sucesso!")
else:
    print("✅ Arquivo ZIP já presente no ambiente!")

# 3. Descompacta rapidamente com o unzip nativo do Linux
if not os.path.exists(PASTA_EXTRACAO):
    print("📦 Descompactando arquivos (aguarde)...")
    !unzip -q /content/UFPR-ALPR.zip -d /content/ufpr_raw
    print("✅ Descompactação finalizada com sucesso!")
else:
    print("✅ Pasta descompactada já existe!")

## 4. Conversão das Anotações da UFPR para o Formato YOLO (Robusta & Automática)
Varre recursivamente todas as anotações do UFPR-ALPR, converte as caixas da placa para o padrão normalizado YOLO `(x_center, y_center, width, height)` e divide em 80% Treino e 20% Validação.

In [ ]:
import os
import glob
import shutil
import random
import cv2

DATASET_YOLO = "/content/dataset_ufpr_yolo"

# Limpa pasta anterior se houver e recria estrutura YOLO
if os.path.exists(DATASET_YOLO):
    shutil.rmtree(DATASET_YOLO)

os.makedirs(os.path.join(DATASET_YOLO, "train/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "train/labels"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/images"), exist_ok=True)
os.makedirs(os.path.join(DATASET_YOLO, "val/labels"), exist_ok=True)

print("🔍 Buscando arquivos de anotação do UFPR-ALPR...")
todos_txt = glob.glob("/content/ufpr_raw/**/*.txt", recursive=True)
if not todos_txt:
    todos_txt = glob.glob("/content/**/*.txt", recursive=True)

pares_validos = []
for txt_path in todos_txt:
    # Procura imagem correspondente
    img_path = txt_path.rsplit('.', 1)[0] + ".png"
    if not os.path.exists(img_path):
        img_path = txt_path.rsplit('.', 1)[0] + ".jpg"
        if not os.path.exists(img_path):
            continue
            
    # Lê arquivo txt para ver se tem anotação da placa
    try:
        with open(txt_path, 'r', encoding='utf-8', errors='ignore') as f:
            linhas = f.readlines()
    except Exception:
        continue
        
    boxes_placa = []
    for linha in linhas:
        if "position_plate:" in linha.lower():
            partes = linha.strip().split(":")[1].strip().split()
            if len(partes) >= 4:
                x, y, w, h = map(float, partes[:4])
                boxes_placa.append((x, y, w, h))
                
    if boxes_placa:
        pares_validos.append((img_path, txt_path, boxes_placa))

print(f"✅ Total de {len(pares_validos)} imagens com placas anotadas encontradas!")

# Embaralha e divide em 80% Treino e 20% Validação
random.seed(42)
random.shuffle(pares_validos)
corte = int(len(pares_validos) * 0.8)
split_treino = pares_validos[:corte]
split_val = pares_validos[corte:]

def salvar_amostras(amostras, split_nome):
    for img_path, txt_path, boxes in amostras:
        img = cv2.imread(img_path)
        if img is None:
            continue
        h_img, w_img = img.shape[:2]
        
        yolo_labels = []
        for x, y, w, h in boxes:
            x_center = (x + w / 2.0) / w_img
            y_center = (y + h / 2.0) / h_img
            norm_w = w / w_img
            norm_h = h / h_img
            yolo_labels.append(f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f}")
            
        nome_base = os.path.basename(img_path)
        shutil.copy(img_path, os.path.join(DATASET_YOLO, f"{split_nome}/images", nome_base))
        
        nome_txt = os.path.basename(img_path).rsplit('.', 1)[0] + ".txt"
        with open(os.path.join(DATASET_YOLO, f"{split_nome}/labels", nome_txt), 'w') as f_out:
            f_out.write("\n".join(yolo_labels))

print("📁 Gravando conjunto de Treino...")
salvar_amostras(split_treino, "train")
print(f"✅ Treino: {len(split_treino)} imagens salvas.")

print("📁 Gravando conjunto de Validação...")
salvar_amostras(split_val, "val")
print(f"✅ Validação: {len(split_val)} imagens salvas.")

# Cria arquivo data.yaml
yaml_content = f"""
path: {DATASET_YOLO}
train: train/images
val: val/images

names:
  0: placa
"""

yaml_path = os.path.join(DATASET_YOLO, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(f"\n🎉 data.yaml pronto para o YOLO: {yaml_path}")

## 5. Treinamento do YOLO com Dataset Brasileiro UFPR-ALPR

In [ ]:
from ultralytics import YOLO

modelo = YOLO("yolov8n.pt")

resultados = modelo.train(
    data=os.path.join(DATASET_YOLO, "data.yaml"),
    epochs=40,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    name="yolo_ufpr_placas_brasil"
)

print("🎉 Treinamento com UFPR-ALPR finalizado com sucesso!")

## 6. Avaliação das Métricas e Desempenho

In [ ]:
from IPython.display import Image, display
import glob

for grafico in glob.glob("runs/detect/yolo_ufpr_placas_brasil/*.png"):
    print(f"📊 Gráfico: {grafico}")
    display(Image(filename=grafico))
    print("-" * 50)

## 7. Download dos Melhores Pesos (`best.pt`)
Baixe o arquivo `best.pt` e coloque-o na pasta `models/best.pt` do seu projeto local.

In [ ]:
from google.colab import files

caminho_pesos = "runs/detect/yolo_ufpr_placas_brasil/weights/best.pt"
if os.path.exists(caminho_pesos):
    print("⬇️ Baixando modelo treinado com UFPR-ALPR...")
    files.download(caminho_pesos)
else:
    print("❌ Arquivo best.pt não encontrado.")